In [41]:
from datetime import datetime
from pydantic import BaseModel, PositiveInt

In [42]:
class User(BaseModel):
  # type hint(annotation)
  id: int
  name: str = 'John Doe'
  signup_ts: datetime | None
  tastes: dict[str, PositiveInt]

In [43]:
external_data = {
    'id': 123,
    'name': 'Kevin',
    'signup_ts': '2026-09-07 12:34',
    'tastes': {
        'wine': 9,
        b'cheese': 7, # pydantic이 자동으로 bin 타입을 str으로 변환해줌.
        'cabbage': '1',
    },
}

In [44]:
# User 오브젝트 생성
user = User(**external_data) # **은 딕셔너리 언패킹

In [45]:
print(user.id)
print(user.model_dump())

123
{'id': 123, 'name': 'Kevin', 'signup_ts': datetime.datetime(2026, 9, 7, 12, 34), 'tastes': {'wine': 9, 'cheese': 7, 'cabbage': 1}}


In [46]:
from datetime import datetime
from pydantic import BaseModel, PositiveInt, ValidationError

class User(BaseModel):
  id: int
  name: str = 'John Doe'
  signup_ts: datetime | None
  tastes: dict[str, PositiveInt]

external_data = {'id': 'not an int', 'tastes': {}}

try:
  User(**external_data)
except ValidationError as e:
  print(e.errors())

[{'type': 'int_parsing', 'loc': ('id',), 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'not an int', 'url': 'https://errors.pydantic.dev/2.13/v/int_parsing'}, {'type': 'missing', 'loc': ('signup_ts',), 'msg': 'Field required', 'input': {'id': 'not an int', 'tastes': {}}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}]


In [47]:
from typing import Annotated, Literal

from annotated_types import Gt

from pydantic import BaseModel


class Fruit(BaseModel):
  name: str
  color: Literal['red', 'green']
  weight: Annotated[float, Gt(0)]
  bazam: dict[str, list[tuple[int, bool, float]]]


print(
    Fruit(
        name='Apple',
        color='red',
        weight=4.2,
        bazam={'footbar': [(1, True, 0.1)]}
    )
)

name='Apple' color='red' weight=4.2 bazam={'footbar': [(1, True, 0.1)]}


In [48]:
from datetime import datetime
from pydantic import BaseModel

class Meeting(BaseModel):
  when: datetime
  where: bytes
  why: str = 'No idea'

m = Meeting(when='2020-01-01T12:00', where='home')
print(m.model_dump(exclude_unset=True))
print(m.model_dump(exclude={'where'}, mode='json'))
print(m.model_dump_json(exclude_defaults=True))

{'when': datetime.datetime(2020, 1, 1, 12, 0), 'where': b'home'}
{'when': '2020-01-01T12:00:00', 'why': 'No idea'}
{"when":"2020-01-01T12:00:00","where":"home"}


In [49]:
from datetime import datetime
from pydantic import BaseModel

class Address(BaseModel):
  street: str
  city: str
  zipcode: str


class Meeting(BaseModel):
  when: datetime
  where: Address
  why: str = 'No idea'

print(Meeting.model_json_schema())

{'$defs': {'Address': {'properties': {'street': {'title': 'Street', 'type': 'string'}, 'city': {'title': 'City', 'type': 'string'}, 'zipcode': {'title': 'Zipcode', 'type': 'string'}}, 'required': ['street', 'city', 'zipcode'], 'title': 'Address', 'type': 'object'}}, 'properties': {'when': {'format': 'date-time', 'title': 'When', 'type': 'string'}, 'where': {'$ref': '#/$defs/Address'}, 'why': {'default': 'No idea', 'title': 'Why', 'type': 'string'}}, 'required': ['when', 'where'], 'title': 'Meeting', 'type': 'object'}


In [50]:
from datetime import datetime
from pydantic import BaseModel, ValidationError
class Meeting(BaseModel):
    when: datetime
    where: bytes
m = Meeting.model_validate({'when': '2020-01-01T12:00', 'where': 'home'})
print(m)
#> when=datetime.datetime(2020, 1, 1, 12, 0) where=b'home'
try:
    m = Meeting.model_validate(
        {'when': '2020-01-01T12:00', 'where': 'home'}, strict=True
    )
except ValidationError as e:
    print(e)
    """
    2 validation errors for Meeting
    when
      Input should be a valid datetime [type=datetime_type, input_value='2020-01-01T12:00', input_type=str]
    where
      Input should be a valid bytes [type=bytes_type, input_value='home', input_type=str]
    """
m_json = Meeting.model_validate_json(
    '{"when": "2020-01-01T12:00", "where": "home"}'
)
print(m_json)
#> when=datetime.datetime(2020, 1, 1, 12, 0) where=b'home'

when=datetime.datetime(2020, 1, 1, 12, 0) where=b'home'
2 validation errors for Meeting
when
  Input should be a valid datetime [type=datetime_type, input_value='2020-01-01T12:00', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/datetime_type
where
  Input should be a valid bytes [type=bytes_type, input_value='home', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/bytes_type
when=datetime.datetime(2020, 1, 1, 12, 0) where=b'home'


In [53]:
from pydantic import BaseModel, ConfigDict

class User(BaseModel):
    id: int
    name: str = 'Jane Doe'

    model_config = ConfigDict(str_max_length=10)


user = User(id='123')
print(user)
print(type(user))
print(type(user.id), user.id)
print(type(user.name), user.name)

id=123 name='Jane Doe'
<class '__main__.User'>
<class 'int'> 123
<class 'str'> Jane Doe
